# 08 — API Client Testing

> **Audience:** Anyone who wants to use the PBI API client for fast, shared
> database access without loading the full SequenceRetriever locally.

| | |
|---|---|
| **Prerequisites** | PBI API service running (`docker compose up api`)
| **Requirements** | `pip install pbi requests pandas` |

## What this notebook covers

1. Connect to the PBI API via `APIClient`
2. Metadata queries: phage, host, phage-host, protein
3. Single sequence and genome retrieval
4. Arbitrary SQL queries
5. Comparison with direct `quick_connect()` approach

## Setup

In [7]:
import sys
from pathlib import Path

# Ensure pbi package is importable
project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pbi
from pbi import APIClient

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 20)

print(f"pbi version: {pbi.__version__}")
print(f"APIClient available: {hasattr(pbi, 'APIClient')}")

pbi version: 0.3.0
APIClient available: True


## 1. Connect to the API

The `APIClient` communicates with the PBI API service over HTTP.
Make sure the API is running: `docker compose up api`.

In [8]:
# Adjust the URL if the API is running elsewhere
# Inside Docker: http://pbi-api:8000
# Outside Docker: http://localhost:8000
import os
API_URL = os.getenv("PBI_API_URL", "http://pbi-api:8000")

client = APIClient(API_URL)

# Health check
health = client.health()
print(f"API health: {health}")

API health: {'status': 'healthy', 'database': 'connected'}


In [9]:
# Database statistics
stats = client.get_stats()
print("Database statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

Database statistics:
  database: {'phages': 1350644, 'proteins': 71971209, 'source_breakdown': [{'source_type': 'private', 'Source_DB': 'test_private', 'count': 5}, {'source_type': 'private', 'Source_DB': 'test_private_2', 'count': 3}, {'source_type': 'public', 'Source_DB': 'MetaVR', 'count': 225269}, {'source_type': 'public', 'Source_DB': 'GOV2', 'count': 195699}, {'source_type': 'public', 'Source_DB': 'MGV', 'count': 189680}, {'source_type': 'public', 'Source_DB': 'IMGVR', 'count': 177361}, {'source_type': 'public', 'Source_DB': 'GPD', 'count': 142809}, {'source_type': 'public', 'Source_DB': 'TemPhD', 'count': 66823}, {'source_type': 'public', 'Source_DB': 'ELGV', 'count': 56716}, {'source_type': 'public', 'Source_DB': 'CHVD', 'count': 44935}, {'source_type': 'public', 'Source_DB': 'URPC', 'count': 44633}, {'source_type': 'public', 'Source_DB': 'TYMEFLIES', 'count': 44192}, {'source_type': 'public', 'Source_DB': 'OVD', 'count': 37184}, {'source_type': 'public', 'Source_DB': 'GVD', 'c

## 2. Metadata Queries

### 2.1 Phage Metadata

In [10]:
# Get all phage metadata (limited)
phage_meta = client.get_phage_metadata(limit=100)
print(f"Phage metadata: {len(phage_meta)} rows")
phage_meta.head()

Phage metadata: 100 rows


,Phage_ID,Source_DB,Length,GC_content,Taxonomy,Completeness,Host,Lifestyle,Cluster,Subcluster
0,NC_001330.1,RefSeq,6087,45.178249,Microviridae,High-quality,Escherichia coli,virulent,cluster_158413,subcluster_199459
1,NC_001331.1,RefSeq,7349,61.491359,Inoviridae,Low-quality,Pseudomonas aeruginosa,temperate,cluster_67555,subcluster_85085
2,NC_001332.1,RefSeq,6744,42.719454,Inoviridae,Medium-quality,Escherichia coli,virulent,cluster_63254,subcluster_79716
3,NC_001335.1,RefSeq,52297,62.257873,Caudovirales,High-quality,Mycobacterium smegmatis,temperate,cluster_272329,subcluster_342024
4,NC_001341.1,RefSeq,4491,33.288800,None,Not-determined,Acholeplasma laidlawii,virulent,cluster_229441,subcluster_288779


In [11]:
# Filtered query
refseq_phages = client.get_phage_metadata(
    where_clause="Source_DB = 'RefSeq' AND Length > 50000",
    limit=50
)
print(f"RefSeq phages >50kb: {len(refseq_phages)} rows")
refseq_phages.head()

RefSeq phages >50kb: 50 rows


,Phage_ID,Source_DB,Length,GC_content,Taxonomy,Completeness,Host,Lifestyle,Cluster,Subcluster
0,NC_001335.1,RefSeq,52297,62.257873,Caudovirales,High-quality,Mycobacterium smegmatis,temperate,cluster_272329,subcluster_342024
1,NC_001884.1,RefSeq,134416,34.635014,Caudovirales,High-quality,Bacillus subtilis,temperate,cluster_8044,subcluster_10000
2,NC_000924.1,RefSeq,61670,49.367602,Caudovirales,High-quality,Escherichia coli,temperate,cluster_152435,subcluster_191866
3,NC_000902.1,RefSeq,60942,49.913032,Caudovirales,High-quality,Escherichia coli,temperate,cluster_152435,subcluster_191867
4,NC_002656.1,RefSeq,50550,63.649852,Caudovirales,High-quality,Mycobacterium smegmatis,temperate,cluster_8045,subcluster_10001


### 2.2 Host Metadata

In [12]:
host_meta = client.get_host_metadata(limit=50)
print(f"Host metadata: {len(host_meta)} rows")
host_meta.head()

HTTPError: 500 Server Error: Internal Server Error for url: http://pbi-api:8000/host-metadata?limit=50

### 2.3 Phage-Host Metadata

In [ ]:
pairs_meta = client.get_phage_host_metadata(limit=50)
print(f"Phage-host pairs: {len(pairs_meta)} rows")
pairs_meta.head()

### 2.4 Protein Metadata

In [ ]:
protein_meta = client.get_protein_metadata(limit=50)
print(f"Protein metadata: {len(protein_meta)} rows")
protein_meta.head()

## 3. Sequence Retrieval

In [ ]:
# Single phage sequence
phage_id = refseq_phages["Phage_ID"].iloc[0] if len(refseq_phages) > 0 else "NC_001330.1"
seq = client.get_phage_sequence(phage_id)
if seq:
    print(f">{phage_id}")
    print(f"Length: {len(seq):,} bp")
    print(f"First 100 bp: {seq[:100]}...")
else:
    print(f"Sequence not found for {phage_id}")

## 4. Genome Retrieval

In [ ]:
# Phage genome (concatenated)
genome = client.get_phage_genome(phage_id, mode="concat")
if genome:
    print(f">{phage_id} (concat)")
    print(f"Length: {len(genome):,} bp")
else:
    print(f"Genome not found for {phage_id}")

In [ ]:
# Host genome stats
if len(host_meta) > 0:
    host_id = host_meta["Host_ID"].iloc[0]
    h_stats = client.get_host_genome_stats(host_id)
    print(f"Host genome stats for {host_id}:")
    if h_stats:
        for k, v in h_stats.items():
            print(f"  {k}: {v}")
    else:
        print("  No stats available")
else:
    print("No host data available")

## 5. Arbitrary SQL Queries

In [ ]:
# Source database distribution
source_dist = client.query("""
    SELECT
        Source_DB,
        COUNT(*) AS phage_count,
        ROUND(AVG(Length), 0) AS avg_length,
        ROUND(AVG(GC_content), 2) AS avg_gc
    FROM fact_phages
    GROUP BY Source_DB
    ORDER BY phage_count DESC
")
print("Source database distribution:")
source_dist

In [ ]:
# Lifestyle distribution
lifestyle = client.query("""
    SELECT
        COALESCE(Lifestyle, 'Unknown') AS Lifestyle,
        COUNT(*) AS count
    FROM fact_phages
    GROUP BY Lifestyle
    ORDER BY count DESC
")
print("Lifestyle distribution:")
lifestyle

In [ ]:
# Visualize source distribution
if len(source_dist) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(source_dist["Source_DB"], source_dist["phage_count"],
                color=sns.color_palette("Set2", len(source_dist)))
    axes[0].set_xlabel("Source Database")
    axes[0].set_ylabel("Phage Count")
    axes[0].set_title("Phages by Source Database")
    axes[0].tick_params(axis="x", rotation=30)

    axes[1].pie(source_dist["phage_count"], labels=source_dist["Source_DB"],
                autopct="%1.1f%%", colors=sns.color_palette("Set2", len(source_dist)))
    axes[1].set_title("Proportion by Source Database")

    plt.suptitle("Source Database Distribution (via API)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 6. Table Listing

In [ ]:
tables = client.list_tables()
print(f"Tables and views ({len(tables)}):")
tables

## 7. Cleanup

In [ ]:
client.close()
print("API client closed.")